# Notebook v4 — Detección de deforestación (versión de alto rendimiento)

**Objetivo:** F1 0.60-0.70 mediante más datos + técnicas de desbalance.

## Cambios clave vs v3

| # | Cambio | Por qué |
|---|---|---|
| 1 | **4 AOIs** (Ucayali, Madre de Dios, San Martín, Loreto) | Factor #1 según literatura: más datos = más F1. Pasa de ~300 a ~3,000-5,000 patches |
| 2 | **Focal+Dice Loss** | Focal está diseñada para desbalance extremo, mejor que BCE |
| 3 | **Oversampling de positivos** | El modelo ve más ejemplos de deforestación sin perder contexto |
| 4 | **Attention U-Net** | Ya demostró superar baseline en v3 |
| 5 | Todo en disco local `/content/` | Evita timeouts de Drive |

## Flujo de ejecución

1. **Parte A** (celdas 1-9): exporta y descarga las 4 zonas → genera ~3,000-5,000 patches
2. **Pausa**: las descargas de GEE tardan; este notebook descarga directo (sin Drive async)
3. **Parte B** (celdas 10-20): entrena con Focal+Dice + oversampling

⚠ Tiempo estimado total: ~2-3 horas (descarga 4 zonas + entrenamiento). Mantén Colab abierto.

---

# PARTE A — Extracción multi-zona

## Celda 1 — Setup

In [ ]:
import ee

# Esto abre el navegador para elegir cuenta manualmente
ee.Authenticate(force=True, auth_mode='notebook')

In [ ]:
import ee
ee.Initialize(project='feisty-mason-495316-e6')
print('GEE inicializado correctamente')

In [ ]:
import ee

# Fuerza autenticación manual — abrirá un link para elegir tu cuenta
ee.Authenticate(force=True)
ee.Initialize(project='feisty-mason-495316-e6')
print('GEE inicializado')

In [ ]:
!pip install -q earthengine-api geemap rasterio

import ee, geemap
import os, glob, time, random, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

GEE_PROJECT = 'feisty-mason-495316-e6'
try:
    ee.Initialize(project=GEE_PROJECT)
    print('GEE inicializado')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE autenticado e inicializado')

DIR_LOCAL = '/content'
DIR_PARCHES = f'{DIR_LOCAL}/parches'
DIR_RESULTADOS = f'{DIR_LOCAL}/resultados'
os.makedirs(DIR_RESULTADOS, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ ADVERTENCIA: No hay GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## Celda 2 — Definición de las 4 zonas de deforestación activa

Estas zonas fueron elegidas por ser focos conocidos de deforestación en la Amazonía peruana, lo que garantiza más píxeles positivos por zona.

In [ ]:
# 4 AOIs en focos de deforestación activa de la Amazonía peruana
AOIS = {
    'ucayali':      ee.Geometry.Rectangle([-75.0, -8.8, -74.0, -7.8]),   # original
    'madre_dios':   ee.Geometry.Rectangle([-70.5, -12.8, -69.5, -11.8]), # minería + agricultura
    'san_martin':   ee.Geometry.Rectangle([-77.0, -7.0, -76.0, -6.0]),   # frontera agrícola
    'loreto_sur':   ee.Geometry.Rectangle([-74.5, -5.5, -73.5, -4.5]),   # expansión reciente
}

FECHA_T1_INI, FECHA_T1_FIN = '2021-06-01', '2021-09-30'
FECHA_T2_INI, FECHA_T2_FIN = '2022-06-01', '2022-09-30'

# Visualizar las zonas
Map = geemap.Map(center=[-8.0, -74.0], zoom=6)
colores = ['red', 'blue', 'green', 'orange']
for (nombre, geom), color in zip(AOIS.items(), colores):
    Map.addLayer(geom, {'color': color}, nombre)
Map

## Celda 3 — Funciones de procesamiento (mosaicos, índices, etiquetas)

In [ ]:
def enmascarar_nubes_s2(img):
    scl = img.select('SCL')
    mascara = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                .And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(mascara).divide(10000).copyProperties(img, ['system:time_start'])

def construir_mosaico(fecha_ini, fecha_fin, aoi):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi).filterDate(fecha_ini, fecha_fin)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
           .map(enmascarar_nubes_s2))
    return col.median().clip(aoi)

def calcular_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    bsi = img.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
        {'SWIR1': img.select('B11'), 'RED': img.select('B4'),
         'NIR': img.select('B8'), 'BLUE': img.select('B2')}
    ).rename('BSI')
    return img.addBands([ndvi, bsi])

def construir_etiqueta_mapbiomas(aoi):
    MB_ASSET = 'projects/mapbiomas-raisg/public/collection5/mapbiomas_raisg_panamazonia_collection5_integration_v1'
    mb = ee.Image(MB_ASSET)
    cls_2021 = mb.select('classification_2021').clip(aoi)
    cls_2022 = mb.select('classification_2022').clip(aoi)
    bosque_2021 = cls_2021.eq(3).Or(cls_2021.eq(6))
    bosque_2022 = cls_2022.eq(3).Or(cls_2022.eq(6))
    etiqueta = bosque_2021.And(bosque_2022.Not()).rename('label_mb').toByte()
    return etiqueta.reproject(crs='EPSG:4326', scale=10).unmask(0)

def construir_stack(aoi):
    m_t1 = calcular_indices(construir_mosaico(FECHA_T1_INI, FECHA_T1_FIN, aoi))
    m_t2 = calcular_indices(construir_mosaico(FECHA_T2_INI, FECHA_T2_FIN, aoi))
    etiqueta = construir_etiqueta_mapbiomas(aoi)
    bandas = ['B2', 'B3', 'B4', 'B8', 'NDVI', 'BSI']
    stack = (m_t1.select(bandas).rename([f'{b}_t1' for b in bandas])
             .addBands(m_t2.select(bandas).rename([f'{b}_t2' for b in bandas]))
             .addBands(etiqueta))
    return stack.toFloat()

print('Funciones listas')

## Celda 4 — Descargar las 4 zonas a disco local

Descarga directa con geemap (sin Drive). Cada zona tarda ~5-15 min. Total ~30-60 min.

In [ ]:
!pip install geedim -q

In [ ]:
rutas_tif = {}
for nombre, geom in AOIS.items():
    ruta = f'{DIR_LOCAL}/stackkk_{nombre}.tif'
    rutas_tif[nombre] = ruta
    if os.path.exists(ruta):
        print(f'✓ {nombre} ya descargado ({os.path.getsize(ruta)/1024/1024:.0f} MB)')
        continue
    print(f'Descargando {nombre}...')
    t0 = time.time()
    try:
        stack = construir_stack(geom)
        geemap.download_ee_image(
            image=stack, filename=ruta,
            region=geom, scale=10, crs='EPSG:4326'
        )
        print(f'  ✓ {(time.time()-t0)/60:.1f} min ({os.path.getsize(ruta)/1024/1024:.0f} MB)')
    except Exception as e:
        print(f'  ✗ Error en {nombre}: {e}')

print('\nDescargas completas')

In [ ]:
!rm -rf /content/parches



## Celda 5 — Generar parches de todas las zonas

Split geográfico DENTRO de cada zona (70/15/15) para evitar leakage. Conserva solo parches con >2% de deforestación para mantener balance.

In [ ]:
import rasterio
from rasterio.windows import Window

TAM_PARCHE = 256
STRIDE = 128          # solapamiento 50%
MIN_PCT_POS = 0.3     # solo parches con >2% de deforestacion
LABEL_IDX = 12
BORRAR_TIF = True     # <-- borra el .tif de cada zona EN CUANTO termina sus parches

for sp in ['train','val','test']:
    os.makedirs(f'{DIR_PARCHES}/{sp}', exist_ok=True)

cont_total = {'train':0,'val':0,'test':0,'descartados':0}

for nombre, ruta_tif in rutas_tif.items():
    if not os.path.exists(ruta_tif):
        print(f'⚠ {nombre}: no hay .tif (¿ya borrado?), saltando'); continue

    cont_zona = {'train':0,'val':0,'test':0,'descartados':0}
    with rasterio.open(ruta_tif) as src:
        H, W = src.height, src.width
        fila_train_fin = int(H*0.70); fila_val_fin = int(H*0.85)
        for fila in range(0, H-TAM_PARCHE+1, STRIDE):
            for col in range(0, W-TAM_PARCHE+1, STRIDE):
                parche = src.read(window=Window(col, fila, TAM_PARCHE, TAM_PARCHE)).astype(np.float32)
                # FIX nodata: sentinel -3.4e38 -> NaN (descarte y saneo correctos)
                parche = np.where(np.abs(parche) > 1e30, np.nan, parche)
                if np.isnan(parche[:12]).mean() > 0.20:
                    cont_zona['descartados'] += 1; continue
                if fila+TAM_PARCHE <= fila_train_fin: split='train'
                elif fila+TAM_PARCHE <= fila_val_fin: split='val'
                else: split='test'
                etiqueta = parche[LABEL_IDX]
                if np.nanmean(etiqueta)*100 < MIN_PCT_POS:
                    cont_zona['descartados'] += 1; continue
                pf = np.concatenate([parche[:12], etiqueta[None,:,:]], axis=0)
                pf = np.nan_to_num(pf, nan=0.0).astype(np.float32)
                np.save(f'{DIR_PARCHES}/{split}/{nombre}_{fila:06d}_{col:06d}.npy', pf)
                cont_zona[split] += 1
    print(f'{nombre}: train {cont_zona["train"]} | val {cont_zona["val"]} | test {cont_zona["test"]} | descartados {cont_zona["descartados"]}')

    # Borrar el .tif de ESTA zona ya que sus parches quedaron listos -> libera ~5 GB
    if BORRAR_TIF and os.path.exists(ruta_tif):
        os.remove(ruta_tif)
        print(f'  \U0001f5d1 {os.path.basename(ruta_tif)} borrado (disco liberado)')

    for k in cont_total: cont_total[k] += cont_zona[k]

print('\n=== TOTAL ===')
for k,v in cont_total.items(): print(f'  {k}: {v}')


## Celda 6 — Verificar balance del dataset combinado

In [ ]:
for split in ['train', 'val', 'test']:
    rutas = sorted(glob.glob(f'{DIR_PARCHES}/{split}/*.npy'))
    if not rutas:
        print(f'{split}: vacío'); continue
    pcts = [np.load(r, allow_pickle=False)[12].mean()*100 for r in rutas]
    print(f'{split}: {len(rutas)} parches | %pos promedio: {np.mean(pcts):.2f}% | máx: {np.max(pcts):.1f}%')

## Celda 7 — Inspección visual (QA)

In [ ]:
parches_train = sorted(glob.glob(f'{DIR_PARCHES}/train/*.npy'))
ejemplos = []
for p in random.sample(parches_train, min(80, len(parches_train))):
    arr = np.load(p)
    if arr[12].mean() > 0.05:
        ejemplos.append(p)
    if len(ejemplos) >= 3: break

if ejemplos:
    fig, axes = plt.subplots(len(ejemplos), 4, figsize=(16, 4*len(ejemplos)))
    if len(ejemplos) == 1: axes = axes[None, :]
    for i, ruta in enumerate(ejemplos):
        arr = np.load(ruta)
        rgb_t2 = np.clip(np.stack([arr[8], arr[7], arr[6]], axis=-1)/0.3, 0, 1)
        ndvi_diff = arr[10] - arr[4]
        label = arr[12]
        axes[i,0].imshow(np.clip(np.stack([arr[2],arr[1],arr[0]],axis=-1)/0.3,0,1)); axes[i,0].set_title('RGB T1')
        axes[i,1].imshow(rgb_t2); axes[i,1].set_title('RGB T2')
        axes[i,2].imshow(ndvi_diff, cmap='RdYlGn', vmin=-0.5, vmax=0.5); axes[i,2].set_title('ΔNDVI')
        axes[i,3].imshow(label, cmap='Reds'); axes[i,3].set_title(f'Etiqueta ({label.mean()*100:.1f}%)')
        for ax in axes[i]: ax.axis('off')
    plt.tight_layout(); plt.show()

---
# PARTE B — Entrenamiento con Focal+Dice + Oversampling

## Celda 8 — Dataset con oversampling de positivos

In [ ]:
import glob
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'/content/parches/{split}/*.npy'))
    print(f'{split}: {n} parches')

In [ ]:
class DatasetDeforestacion(Dataset):
    def __init__(self, rutas, media=None, std=None, augment=False):
        self.rutas = rutas
        self.media = media; self.std = std; self.augment = augment
    def __len__(self): return len(self.rutas)
    def __getitem__(self, idx):
        for offset in range(len(self.rutas)):
            try:
                ruta = self.rutas[(idx + offset) % len(self.rutas)]
                arr = np.load(ruta, allow_pickle=False)
                if arr.shape != (13, 256, 256): continue
                break
            except Exception: continue
        else:
            raise RuntimeError('Ningún parche válido')

        x = arr[:12].astype(np.float32)
        y = arr[12].astype(np.float32)
        # FIX nodata: sentinel de geedim (-3.4e38) -> NaN -> tras normalizar se mapea a 0 (la media),
        # en vez de desbordar a -inf y quedar saturado a -10 (valor extremo que ensuciaba el entrenamiento).
        x = np.where(np.abs(x) > 1e30, np.nan, x)
        if self.media is not None:
            x = (x - self.media[:, None, None]) / (self.std[:, None, None] + 1e-6)
            x = np.clip(x, -10.0, 10.0)
            x = np.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        if self.augment:
            if random.random() < 0.5: x = x[:, :, ::-1].copy(); y = y[:, ::-1].copy()
            if random.random() < 0.5: x = x[:, ::-1, :].copy(); y = y[::-1, :].copy()
            k = random.randint(0, 3)
            x = np.rot90(x, k, axes=(1,2)).copy(); y = np.rot90(y, k).copy()
        return torch.from_numpy(x), torch.from_numpy(y).unsqueeze(0)


def oversamplear(rutas, umbral_pct=8.0, multiplicador=3):
    """Duplica parches con alta densidad de deforestación."""
    extra = []
    for r in rutas:
        try:
            if np.load(r, allow_pickle=False)[12].mean()*100 >= umbral_pct:
                extra.extend([r] * (multiplicador - 1))
        except: pass
    return rutas + extra


rutas_train = sorted(glob.glob(f'{DIR_PARCHES}/train/*.npy'))
rutas_val   = sorted(glob.glob(f'{DIR_PARCHES}/val/*.npy'))
rutas_test  = sorted(glob.glob(f'{DIR_PARCHES}/test/*.npy'))

# Aplicar oversampling solo a train
rutas_train_os = oversamplear(rutas_train, umbral_pct=8.0, multiplicador=3)
print(f'Train original: {len(rutas_train)} → con oversampling: {len(rutas_train_os)}')
print(f'Val: {len(rutas_val)} | Test: {len(rutas_test)}')

# Normalización robusta
muestras_validas = []
for r in random.sample(rutas_train, min(300, len(rutas_train))):
    try:
        arr = np.load(r, allow_pickle=False)[:12].astype(np.float32)
        if np.isnan(arr).any() or np.isinf(arr).any(): continue
        if np.abs(arr).max() > 1e6: continue
        muestras_validas.append(arr)
    except: pass

print(f'Muestras válidas: {len(muestras_validas)}')
muestras = np.stack(muestras_validas)
media = np.nanmean(muestras, axis=(0,2,3)).astype(np.float32)
std   = np.nanstd(muestras, axis=(0,2,3)).astype(np.float32)
std   = np.clip(std, a_min=1e-4, a_max=None)
print('Media:', np.round(media, 4))
print('Std:  ', np.round(std, 4))
np.savez(f'{DIR_RESULTADOS}/normalizacion.npz', media=media, std=std)

BATCH = 8
dl_train = DataLoader(DatasetDeforestacion(rutas_train_os, media, std, augment=True),
                      batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
dl_val   = DataLoader(DatasetDeforestacion(rutas_val, media, std, augment=False),
                      batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(DatasetDeforestacion(rutas_test, media, std, augment=False),
                      batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## Celda 9 — Arquitectura Attention U-Net

In [ ]:
class DobleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        return x * self.psi(self.relu(self.W_g(g) + self.W_x(x)))

class AttentionUNet(nn.Module):
    def __init__(self, c_in=12, c_out=1, base=32):
        super().__init__()
        self.e1=DobleConv(c_in,base); self.e2=DobleConv(base,base*2)
        self.e3=DobleConv(base*2,base*4); self.e4=DobleConv(base*4,base*8)
        self.bottleneck=DobleConv(base*8,base*16); self.pool=nn.MaxPool2d(2)
        self.up4=nn.ConvTranspose2d(base*16,base*8,2,stride=2); self.ag4=AttentionGate(base*8,base*8,base*4); self.d4=DobleConv(base*16,base*8)
        self.up3=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.ag3=AttentionGate(base*4,base*4,base*2); self.d3=DobleConv(base*8,base*4)
        self.up2=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.ag2=AttentionGate(base*2,base*2,base); self.d2=DobleConv(base*4,base*2)
        self.up1=nn.ConvTranspose2d(base*2,base,2,stride=2); self.ag1=AttentionGate(base,base,base//2); self.d1=DobleConv(base*2,base)
        self.out=nn.Conv2d(base,c_out,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        u4=self.up4(b); d4=self.d4(torch.cat([u4,self.ag4(u4,e4)],1))
        u3=self.up3(d4); d3=self.d3(torch.cat([u3,self.ag3(u3,e3)],1))
        u2=self.up2(d3); d2=self.d2(torch.cat([u2,self.ag2(u2,e2)],1))
        u1=self.up1(d2); d1=self.d1(torch.cat([u1,self.ag1(u1,e1)],1))
        return self.out(d1)

print(f'AttentionUNet: {sum(p.numel() for p in AttentionUNet().parameters()):,} params')

## Celda 10 — Focal+Dice Loss + métricas

**Focal Loss** penaliza más los píxeles difíciles (positivos raros) y menos los fáciles (negativos abundantes). Es la mejor opción documentada para desbalance extremo en segmentación.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
        pt = torch.exp(-bce)
        # alpha pondera la clase positiva (rara)
        alpha_t = self.alpha * y + (1 - self.alpha) * (1 - y)
        focal = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()

class DiceLoss(nn.Module):
    def __init__(self, eps=1.0):
        super().__init__(); self.eps=eps
    def forward(self, logits, y):
        prob = torch.sigmoid(logits)
        inter = (prob*y).sum(dim=(2,3))
        union = prob.sum(dim=(2,3)) + y.sum(dim=(2,3))
        return 1 - ((2*inter+self.eps)/(union+self.eps)).mean()

class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, dice_eps=1.0, focal_w=0.5, dice_w=0.5):
        super().__init__()
        self.focal=FocalLoss(alpha,gamma); self.dice=DiceLoss(dice_eps)
        self.fw=focal_w; self.dw=dice_w
    def forward(self, logits, y):
        return self.fw*self.focal(logits,y) + self.dw*self.dice(logits,y)

def calc_metricas(logits, y, umbral=0.5):
    pred = (torch.sigmoid(logits) > umbral).float()
    tp=((pred==1)&(y==1)).sum().item(); fp=((pred==1)&(y==0)).sum().item()
    fn=((pred==0)&(y==1)).sum().item(); tn=((pred==0)&(y==0)).sum().item()
    return tp,fp,fn,tn

def metricas_finales(tp,fp,fn,tn,eps=1e-7):
    p=tp/(tp+fp+eps); r=tp/(tp+fn+eps)
    return {'precision':p,'recall':r,'f1':2*p*r/(p+r+eps),
            'iou':tp/(tp+fp+fn+eps),'accuracy':(tp+tn)/(tp+fp+fn+tn+eps)}

print('Losses y métricas listas')

## Celda 11 — Loop de entrenamiento + búsqueda de umbral

In [ ]:
def entrenar(modelo, loss_fn, nombre, epocas=60, lr=1e-3):
    modelo = modelo.to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    # Scheduler simple y estable
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=5, min_lr=1e-6
    )
    hist = {'train_loss':[], 'val_loss':[], 'val_iou':[], 'val_f1':[]}
    mejor_iou = -1
    ruta = f'{DIR_RESULTADOS}/{nombre}_best.pt'

    for ep in range(epocas):
        # TRAIN
        modelo.train(); la=0; n=0; t0=time.time()
        for x, y in dl_train:
            x, y = x.to(device), y.to(device)
            if torch.isnan(x).any(): continue  # salta batches corruptos
            opt.zero_grad()
            logits = modelo(x)
            loss = loss_fn(logits, y)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step()
            la += loss.item() * x.size(0); n += x.size(0)
        tl = la / max(n, 1)

        # VALIDACIÓN
        modelo.eval(); la=0; n=0; TP=FP=FN=TN=0
        with torch.no_grad():
            for x, y in dl_val:
                x, y = x.to(device), y.to(device)
                if torch.isnan(x).any(): continue
                logits = modelo(x)
                loss = loss_fn(logits, y)
                if not torch.isnan(loss):
                    la += loss.item() * x.size(0); n += x.size(0)
                tp,fp,fn_,tn = calc_metricas(logits, y, 0.5)
                TP+=tp; FP+=fp; FN+=fn_; TN+=tn
        vl = la / max(n, 1)
        m = metricas_finales(TP, FP, FN, TN)
        sched.step(m['iou'])

        hist['train_loss'].append(tl)
        hist['val_loss'].append(vl)
        hist['val_iou'].append(m['iou'])
        hist['val_f1'].append(m['f1'])

        print(f"[{nombre}] Ep {ep+1:02d}/{epocas} | tr {tl:.4f} | val {vl:.4f} "
              f"| IoU {m['iou']:.4f} | F1 {m['f1']:.4f} | "
              f"P {m['precision']:.3f} R {m['recall']:.3f} | {time.time()-t0:.0f}s")

        if m['iou'] > mejor_iou:
            mejor_iou = m['iou']
            torch.save(modelo.state_dict(), ruta)
            print(f'  → guardado (IoU {mejor_iou:.4f})')

    with open(f'{DIR_RESULTADOS}/{nombre}_hist.json','w') as f:
        json.dump(hist, f)
    return ruta, hist

def buscar_umbral(modelo, ruta_pesos, nombre):
    umbrales = np.arange(0.05, 0.96, 0.05)
    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()
    L,Y=[],[]
    with torch.no_grad():
        for x,y in dl_val:
            x,y=x.to(device),y.to(device)
            L.append(torch.sigmoid(modelo(x)).cpu()); Y.append(y.cpu())
    probs=torch.cat(L); ys=torch.cat(Y)
    print(f'\n=== {nombre} ===')
    print(f'{"U":>6} {"P":>8} {"R":>8} {"F1":>8} {"IoU":>8}')
    mejor_f1=0; mejor_u=0; tabla=[]
    for u in umbrales:
        pred=(probs>u).float()
        tp=((pred==1)&(ys==1)).sum().item(); fp=((pred==1)&(ys==0)).sum().item()
        fn=((pred==0)&(ys==1)).sum().item(); tn=((pred==0)&(ys==0)).sum().item()
        m=metricas_finales(tp,fp,fn,tn); m['umbral']=float(u); tabla.append(m)
        mk=' ←' if m['f1']>mejor_f1 else ''
        if m['f1']>mejor_f1: mejor_f1=m['f1']; mejor_u=u
        print(f"{u:>6.2f} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['iou']:>8.4f}{mk}")
    return mejor_u, mejor_f1, tabla

print('Funciones de entrenamiento listas')

## Celda 12 — Entrenar Attention U-Net + Focal+Dice

In [ ]:
# Encontrar EXACTAMENTE qué parche produce NaN
import torch, numpy as np

modelo_test = AttentionUNet(c_in=12, c_out=1, base=32).to(device)
modelo_test.eval()

batches_problematicos = 0
with torch.no_grad():
    for i, (x, y) in enumerate(dl_train):
        x = x.to(device)
        logits = modelo_test(x)
        if torch.isnan(logits).any():
            batches_problematicos += 1
            if batches_problematicos == 1:
                print(f'Batch {i}: NaN en logits')
                print(f'  x stats: min={x.min():.3f} max={x.max():.3f} mean={x.mean():.3f}')
                print(f'  x NaN antes: {torch.isnan(x).any()}')
                print(f'  x inf antes: {torch.isinf(x).any()}')

print(f'\nTotal batches problemáticos: {batches_problematicos} de {len(dl_train)}')

In [ ]:
import gc
import time

# Definición de la Loss faltante
class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=None, dice_eps=1.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.dice_eps = dice_eps
        self.dice = DiceLoss(eps=dice_eps)

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pos_weight)
        dice = self.dice(logits, y)
        return 0.5 * bce + 0.5 * dice

# Reset completo de memoria
gc.collect()
torch.cuda.empty_cache()

# Configuración del modelo y loss
pos_weight = torch.tensor([30.0], device=device)
loss_fn = BCEDiceLoss(pos_weight=pos_weight)
modelo = AttentionUNet(c_in=12, c_out=1, base=32)

# Función de entrenamiento estable
def entrenar_seguro(modelo, loss_fn, nombre, epocas=40, lr=3e-4):
    modelo = modelo.to(device)
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    mejor_iou = -1
    ruta = f'{DIR_RESULTADOS}/{nombre}_best.pt'
    hist = {'train_loss':[], 'val_loss':[], 'val_iou':[], 'val_f1':[]}

    for ep in range(epocas):
        modelo.train()
        la = 0; n = 0; t0 = time.time()
        for x, y in dl_train:
            x, y = x.to(device), y.to(device)
            if torch.isnan(x).any(): continue
            opt.zero_grad()
            logits = modelo(x)
            if torch.isnan(logits).any(): continue
            loss = loss_fn(logits, y)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 0.5)
            opt.step()
            la += loss.item() * x.size(0); n += x.size(0)
        tl = la / max(n, 1)

        modelo.eval()
        TP=FP=FN=TN=0; la=0; n=0
        with torch.no_grad():
            for x, y in dl_val:
                x, y = x.to(device), y.to(device)
                logits = modelo(x)
                loss = loss_fn(logits, y)
                if not torch.isnan(loss):
                    la += loss.item()*x.size(0); n += x.size(0)
                tp,fp,fn_,tn = calc_metricas(logits, y, 0.5)
                TP+=tp; FP+=fp; FN+=fn_; TN+=tn
        vl = la / max(n, 1)
        m = metricas_finales(TP,FP,FN,TN)
        sched.step(m['iou'])
        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        hist['val_iou'].append(m['iou']); hist['val_f1'].append(m['f1'])

        print(f"Ep {ep+1:02d}/{epocas} | tr {tl:.4f} | val {vl:.4f} | IoU {m['iou']:.4f} | F1 {m['f1']:.4f} | P {m['precision']:.3f} R {m['recall']:.3f} | {time.time()-t0:.0f}s")
        if m['iou'] > mejor_iou:
            mejor_iou = m['iou']
            torch.save(modelo.state_dict(), ruta)
            print(f"  → Guardado (Mejor IoU: {mejor_iou:.4f})")

    return ruta, hist

ruta_modelo, hist = entrenar_seguro(modelo, loss_fn, 'attn_seguro_v4', epocas=40, lr=3e-4)

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU ACTIVA: {torch.cuda.get_device_name(0)}')
    print(f'Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'Memoria reservada: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB')
    print(f'Memoria asignada: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('❌ GPU NO DISPONIBLE. Ve a: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4')

In [ ]:
import torch
from torch.cuda.amp import GradScaler, autocast

def entrenar_con_amp(modelo, loss_fn, nombre, epocas=40, lr=3e-4):
    modelo = modelo.to(device)
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)

    # Inicializador para escalado de gradientes (necesario para AMP)
    scaler = GradScaler()

    mejor_iou = -1
    ruta = f'{DIR_RESULTADOS}/{nombre}_amp_best.pt'
    hist = {'train_loss':[], 'val_loss':[], 'val_iou':[], 'val_f1':[]}

    for ep in range(epocas):
        modelo.train()
        la = 0; n = 0; t0 = time.time()
        for x, y in dl_train:
            x, y = x.to(device), y.to(device)
            if torch.isnan(x).any(): continue

            opt.zero_grad()

            # Ejecución con precisión mixta
            with autocast():
                logits = modelo(x)
                loss = loss_fn(logits, y)

            if torch.isnan(loss): continue

            # Escalado de gradientes y optimización
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 0.5)
            scaler.step(opt)
            scaler.update()

            la += loss.item() * x.size(0); n += x.size(0)

        tl = la / max(n, 1)

        # Validación (también con autocast para consistencia)
        modelo.eval()
        TP=FP=FN=TN=0; la=0; n=0
        with torch.no_grad():
            for x, y in dl_val:
                x, y = x.to(device), y.to(device)
                with autocast():
                    logits = modelo(x)
                    loss = loss_fn(logits, y)
                if not torch.isnan(loss):
                    la += loss.item()*x.size(0); n += x.size(0)
                tp,fp,fn_,tn = calc_metricas(logits, y, 0.5)
                TP+=tp; FP+=fp; FN+=fn_; TN+=tn

        vl = la / max(n, 1)
        m = metricas_finales(TP,FP,FN,TN)
        sched.step(m['iou'])

        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        hist['val_iou'].append(m['iou']); hist['val_f1'].append(m['f1'])

        print(f"Ep {ep+1:02d}/{epocas} | tr {tl:.4f} | val {vl:.4f} | IoU {m['iou']:.4f} | F1 {m['f1']:.4f} | {time.time()-t0:.0f}s")
        if m['iou'] > mejor_iou:
            mejor_iou = m['iou']
            torch.save(modelo.state_dict(), ruta)
            print(f"  → Guardado con AMP (Mejor IoU: {mejor_iou:.4f})")

    return ruta, hist

# Para usarlo, simplemente descomenta y ejecuta:
# ruta_modelo, hist = entrenar_con_amp(modelo, loss_fn, 'attn_v4_fast', epocas=40)

In [ ]:
import gc

# Limpiar cache antes de empezar
gc.collect()
torch.cuda.empty_cache()

# Iniciar entrenamiento con Precisión Mixta (AMP) para máxima velocidad
print("Iniciando entrenamiento de alto rendimiento...")
ruta_modelo_final, historial = entrenar_con_amp(
    modelo=modelo,
    loss_fn=loss_fn,
    nombre='attn_v4_amp_final',
    epocas=40,
    lr=3e-4
)

## Celda 13 — Búsqueda de umbral óptimo

In [ ]:
umbral_opt, f1_val, tabla = buscar_umbral(
    AttentionUNet(c_in=12, c_out=1, base=32), ruta_modelo, 'Attn+FocalDice'
)
print(f'\n→ Umbral óptimo: {umbral_opt:.2f} | F1 validación: {f1_val:.4f}')

## Celda 14 — Evaluación final en TEST

In [ ]:
def evaluar_test(modelo, ruta_pesos, umbral):
    modelo.load_state_dict(torch.load(ruta_pesos, map_location=device))
    modelo.to(device).eval()
    TP=FP=FN=TN=0
    with torch.no_grad():
        for x,y in dl_test:
            x,y=x.to(device),y.to(device)
            tp,fp,fn_,tn=calc_metricas(modelo(x),y,umbral); TP+=tp;FP+=fp;FN+=fn_;TN+=tn
    return metricas_finales(TP,FP,FN,TN)

m_test = evaluar_test(AttentionUNet(c_in=12,c_out=1,base=32), ruta_modelo, umbral_opt)

print('='*60)
print('RESULTADO FINAL EN TEST (Attention U-Net + Focal+Dice)')
print('='*60)
print(f"  Umbral óptimo: {umbral_opt:.2f}")
print(f"  F1:        {m_test['f1']:.4f}")
print(f"  IoU:       {m_test['iou']:.4f}")
print(f"  Precision: {m_test['precision']:.4f}")
print(f"  Recall:    {m_test['recall']:.4f}")
print(f"  Accuracy:  {m_test['accuracy']:.4f}")

with open(f'{DIR_RESULTADOS}/resultado_final_v4.json','w') as f:
    json.dump({'umbral':umbral_opt, 'test':m_test}, f, indent=2)

## Celda 15 — Visualización de predicciones

In [ ]:
modelo = AttentionUNet(c_in=12,c_out=1,base=32)
modelo.load_state_dict(torch.load(ruta_modelo, map_location=device))
modelo.to(device).eval()

ejemplos=[]
for r in rutas_test:
    try:
        if np.load(r,allow_pickle=False)[12].mean()>0.05: ejemplos.append(r)
        if len(ejemplos)>=4: break
    except: continue

fig,axes=plt.subplots(len(ejemplos),4,figsize=(16,4*len(ejemplos)))
if len(ejemplos)==1: axes=axes[None,:]
with torch.no_grad():
    for i,ruta in enumerate(ejemplos):
        arr=np.load(ruta,allow_pickle=False)
        x_raw=arr[:12]; y=arr[12]
        x_norm=((x_raw-media[:,None,None])/(std[:,None,None]+1e-6)).astype(np.float32)
        prob=torch.sigmoid(modelo(torch.from_numpy(x_norm).unsqueeze(0).to(device)))[0,0].cpu().numpy()
        pred=(prob>umbral_opt).astype(np.float32)
        rgb=np.clip(np.stack([x_raw[8],x_raw[7],x_raw[6]],axis=-1)/0.3,0,1)
        axes[i,0].imshow(rgb); axes[i,0].set_title('RGB T2')
        axes[i,1].imshow(y,cmap='Reds'); axes[i,1].set_title('Ground Truth')
        axes[i,2].imshow(prob,cmap='Reds',vmin=0,vmax=1); axes[i,2].set_title('Probabilidad')
        axes[i,3].imshow(pred,cmap='Reds'); axes[i,3].set_title(f'Predicción (u={umbral_opt:.2f})')
        for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/predicciones_v4.png', dpi=120, bbox_inches='tight')
plt.show()

## Celda 16 — Curvas de aprendizaje

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(20,5))
axes[0].plot(hist['train_loss'],label='train'); axes[0].plot(hist['val_loss'],label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist['val_iou']); axes[1].set_title('IoU validación'); axes[1].set_xlabel('Época'); axes[1].grid(alpha=0.3)
axes[2].plot([m['umbral'] for m in tabla],[m['f1'] for m in tabla],'o-')
axes[2].axvline(umbral_opt,color='r',linestyle=':'); axes[2].set_title('F1 vs umbral')
axes[2].set_xlabel('Umbral'); axes[2].set_ylabel('F1'); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/curvas_v4.png', dpi=120, bbox_inches='tight')
plt.show()

## Celda 17 — Respaldo opcional a Drive

In [ ]:
RESPALDAR = False
if RESPALDAR:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dst = '/content/drive/MyDrive/deforestacion_ucayali/resultados_v4'
    os.makedirs(dst, exist_ok=True)
    for a in os.listdir(DIR_RESULTADOS):
        shutil.copy(f'{DIR_RESULTADOS}/{a}', f'{dst}/{a}')
    print(f'✓ Respaldado a {dst}')
else:
    print('Resultados en', DIR_RESULTADOS)
    for a in sorted(os.listdir(DIR_RESULTADOS)):
        print(' ', a)

---

In [ ]:
# ========================================================================
# DIAGNÓSTICO TECHO MapBiomas vs Hansen — 100% EN GEE
# ------------------------------------------------------------------------
# No requiere: GPU, /content/, modelo .pt, parches, normalización.
# Sólo requiere: autenticación de Google Earth Engine.
#
# Computa F1 entre MapBiomas (deforestación 2021→2022) y Hansen GFC
# (lossyear ∈ {21,22}) en la región de TEST (sur 15% de cada AOI,
# mismo split que tu notebook).
# ========================================================================

import ee

# Si el kernel es nuevo y aún no autenticaste GEE, descomenta:
# ee.Authenticate()
ee.Initialize(project='feisty-mason-495316-e6')

# AOIs del notebook v4
AOIS_BBOX = {
    'ucayali':       (-75.0, -8.8, -74.0, -7.8),
    'madre_de_dios': (-70.5, -12.7, -69.5, -11.7),
    'san_martin':    (-77.2, -7.0, -76.2, -6.0),
    'loreto':        (-75.5, -4.5, -74.5, -3.5),
}

# --- Región de TEST = sur 15% de la AOI (mismo split geográfico) ---
def region_test(bbox, frac_test=0.15):
    lon_min, lat_min, lon_max, lat_max = bbox
    lat_test_max = lat_min + frac_test * (lat_max - lat_min)
    return ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_test_max])

# --- Máscaras de deforestación ---
def mascara_mapbiomas(region):
    MB = 'projects/mapbiomas-raisg/public/collection5/mapbiomas_raisg_panamazonia_collection5_integration_v1'
    mb = ee.Image(MB)
    b21 = mb.select('classification_2021').eq(3).Or(mb.select('classification_2021').eq(6))
    b22 = mb.select('classification_2022').eq(3).Or(mb.select('classification_2022').eq(6))
    return b21.And(b22.Not()).rename('mb').toByte().clip(region).unmask(0)

def mascara_hansen(region, anios=(21, 22)):
    h = ee.Image('UMD/hansen/global_forest_change_2023_v1_11').select('lossyear').clip(region)
    m = h.eq(anios[0])
    for a in anios[1:]:
        m = m.Or(h.eq(a))
    return m.rename('hansen').toByte().unmask(0)

# --- Matriz de confusión vía reduceRegion ---
def matriz_confusion(mb, hansen, region, scale=30):
    tp = mb.And(hansen)
    fp = mb.And(hansen.Not())
    fn = mb.Not().And(hansen)
    tn = mb.Not().And(hansen.Not())
    stack = (tp.rename('tp')
               .addBands(fp.rename('fp'))
               .addBands(fn.rename('fn'))
               .addBands(tn.rename('tn')))
    return stack.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region, scale=scale,
        maxPixels=1e11, bestEffort=True
    ).getInfo()

def metricas(d):
    tp, fp, fn, tn = d['tp'], d['fp'], d['fn'], d['tn']
    total = tp + fp + fn + tn
    p   = tp / (tp + fp + 1e-9)
    r   = tp / (tp + fn + 1e-9)
    f1  = 2*p*r / (p + r + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)
    return dict(prec=p, recall=r, f1=f1, iou=iou,
                pct_mb=(tp+fn)/(total+1e-9)*100,
                pct_hansen=(tp+fp)/(total+1e-9)*100)

# --- Computar por zona y agregado ---
print('='*74)
print('TECHO CROSS-SOURCE — MapBiomas vs Hansen GFC en región de TEST')
print('='*74)
print(f'{"zona":<16}{"F1":>9}{"IoU":>9}{"Prec":>9}{"Recall":>9}{"%MB":>9}{"%Hansen":>10}')
print('-'*74)

agregado = {'tp':0, 'fp':0, 'fn':0, 'tn':0}
for zona, bbox in AOIS_BBOX.items():
    region = region_test(bbox)
    try:
        mb     = mascara_mapbiomas(region)
        hansen = mascara_hansen(region)
        conf   = matriz_confusion(mb, hansen, region, scale=30)
        m      = metricas(conf)
        for k in agregado: agregado[k] += conf[k]
        print(f'{zona:<16}{m["f1"]:>9.4f}{m["iou"]:>9.4f}'
              f'{m["prec"]:>9.4f}{m["recall"]:>9.4f}'
              f'{m["pct_mb"]:>9.3f}{m["pct_hansen"]:>10.3f}')
    except Exception as e:
        print(f'{zona:<16}  ERROR: {str(e)[:50]}')

m_agg = metricas(agregado)
print('-'*74)
print(f'{"AGREGADO":<16}{m_agg["f1"]:>9.4f}{m_agg["iou"]:>9.4f}'
      f'{m_agg["prec"]:>9.4f}{m_agg["recall"]:>9.4f}'
      f'{m_agg["pct_mb"]:>9.3f}{m_agg["pct_hansen"]:>10.3f}')
print('='*74)

print()
print('LECTURA DEL RESULTADO')
print('-'*74)
techo = m_agg["f1"]
f1_modelo = 0.3992   # tu resultado actual (Attn U-Net + Focal+Dice)
print(f'  Tu modelo (test): F1 = {f1_modelo:.4f}')
print(f'  Techo cross-source: F1 = {techo:.4f}')
if techo > 0:
    print(f'  Tu modelo está al {f1_modelo/techo*100:.1f}% del techo teórico\n')

if techo < 0.40:
    print('  ► TECHO MUY BAJO: las fuentes discrepan severamente.')
    print('    Tu F1=0.40 está en o por encima del techo realista.')
    print('    Defensa: la mejora RELATIVA vs baseline (178%) ES el aporte.')
elif techo < 0.60:
    print('  ► TECHO MODERADO: hay margen estructural limitado.')
    print('    Tu F1=0.40 es razonable. TTA/ensemble podrían acercarte al techo.')
else:
    print('  ► TECHO ALTO: las fuentes concuerdan bien.')
    print('    El gap NO se explica por etiquetas. Hay que revisar arquitectura/datos.')

print(f'\n  Ratio %Hansen / %MapBiomas: {m_agg["pct_hansen"]/(m_agg["pct_mb"]+1e-9):.2f}x')
print('  (>1 = Hansen detecta más; <1 = MapBiomas detecta más)')